# L6: Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [5]:
data_analyst_agent = Agent(
    role="Data Analyst",
    goal="Monitor and analyze market data in real-time "
         "to identify trends and predict market movements.",
    backstory="Specializing in financial markets, this agent "
              "uses statistical modeling and machine learning "
              "to provide crucial insights. With a knack for data, "
              "the Data Analyst Agent is the cornerstone for "
              "informing trading decisions.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [7]:
trading_strategy_agent = Agent(
    role="Trading Strategy Developer",
    goal="Develop and test various trading strategies based "
         "on insights from the Data Analyst Agent"
         "focused for novice investors.",
    backstory="Equipped with a deep understanding of financial "
              "markets and quantitative analysis, this agent "
              "devises and refines trading strategies. It evaluates "
              "the performance of different approaches to determine "
              "the most profitable and risk-averse options.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [8]:
execution_agent = Agent(
    role="Trade Advisor",
    goal="Suggest optimal trade execution strategies "
         "based on approved trading strategies.",
    backstory="This agent specializes in analyzing the timing, price, "
              "and logistical details of potential trades. By evaluating "
              "these factors, it provides well-founded suggestions for "
              "when and how trades should be executed to maximize "
              "efficiency and adherence to strategy.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

In [9]:
risk_management_agent = Agent(
    role="Risk Advisor",
    goal="Evaluate and provide insights on the risks "
         "associated with potential trading activities.",
    backstory="Armed with a deep understanding of risk assessment models "
              "and market dynamics, this agent scrutinizes the potential "
              "risks of proposed trades. It offers a detailed analysis of "
              "risk exposure and suggests safeguards to ensure that "
              "trading activities align with the firm’s risk tolerance.",
    verbose=True,
    allow_delegation=True,
    tools = [scrape_tool, search_tool]
)

## Creating Tasks

In [10]:
# Task for Data Analyst Agent: Analyze Market Data
data_analysis_task = Task(
    description=(
        "Continuously monitor and analyze market data for "
        "the selected stock ({stock_selection}). "
        "Use statistical modeling and machine learning to "
        "identify trends and predict market movements."
    ),
    expected_output=(
        "Insights and alerts about significant market "
        "opportunities or threats for {stock_selection}."
    ),
    output_file="DA_report_mid-low_risk.md",  # Outputs the report as a text file
    agent=data_analyst_agent,
)

In [11]:
# Task for Trading Strategy Agent: Develop Trading Strategies
strategy_development_task = Task(
    description=(
        "Develop and refine trading strategies based on "
        "the insights from the Data Analyst and "
        "user-defined risk tolerance ({risk_tolerance}). "
        "Consider trading preferences ({trading_strategy_preference})."
    ),
    expected_output=(
        "A set of potential trading strategies for {stock_selection} "
        "that align with the user's risk tolerance."
    ),
    output_file="strategy_report_mid-low_risk.md",  # Outputs the report as a text file
    agent=trading_strategy_agent,
)


In [12]:
# Task for Trade Advisor Agent: Plan Trade Execution
execution_planning_task = Task(
    description=(
        "Analyze approved trading strategies to determine the "
        "best execution methods for {stock_selection}, "
        "considering current market conditions and optimal pricing."
    ),
    expected_output=(
        "Detailed execution plans suggesting how and when to "
        "execute trades for {stock_selection}."
    ),
    output_file="exec_plan_report_mid-low_risk.md",  # Outputs the report as a text file
    agent=execution_agent,
)


In [13]:
# Task for Risk Advisor Agent: Assess Trading Risks
risk_assessment_task = Task(
    description=(
        "Evaluate the risks associated with the proposed trading "
        "strategies and execution plans for {stock_selection}. "
        "Provide a detailed analysis of potential risks "
        "and suggest mitigation strategies."
    ),
    expected_output=(
        "A comprehensive risk analysis report detailing potential "
        "risks and mitigation recommendations for {stock_selection}."
    ),
    output_file="risk_assement_report_mid-low_risk.md",  # Outputs the report as a text file
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [14]:
from crewai import Crew, Process
from langchain_openai import ChatOpenAI

# Define the crew with agents and tasks
financial_trading_crew = Crew(
    agents=[data_analyst_agent, 
            trading_strategy_agent, 
            execution_agent, 
            risk_management_agent],
    
    tasks=[data_analysis_task, 
           strategy_development_task, 
           execution_planning_task, 
           risk_assessment_task],
    
    manager_llm=ChatOpenAI(model="gpt-3.5-turbo", 
                           temperature=0.7),
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [15]:
# Example data for kicking off the process
financial_trading_inputs = {
    'stock_selection': 'PYPL',
    'initial_capital': '100000',
    'risk_tolerance': 'Medium-Low',
    'trading_strategy_preference': 'Day Trading',
    'news_impact_consideration': True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [16]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

 [DEBUG]: Working Agent: Crew Manager
 [INFO]: Starting Task: Continuously monitor and analyze market data for the selected stock (PYPL). Use statistical modeling and machine learning to identify trends and predict market movements.


> Entering new CrewAgentExecutor chain...
I need to delegate the task of analyzing market data for PYPL to the Data Analyst and ask the Trading Strategy Developer to use statistical modeling and machine learning to identify trends and predict market movements.

Action: Delegate work to co-worker
Action Input: {
    "coworker": "Data Analyst",
    "task": "Continuously monitor and analyze market data for PYPL",
    "context": "The task involves analyzing market data for the stock PYPL using statistical techniques and machine learning to identify trends and predict market movements."
}


> Entering new CrewAgentExecutor chain...
I need to gather real-time market data for PYPL to analyze trends accurately.

Action: Read website content
Action Input: {"websit

Final Answer: PayPal Holdings, Inc. (PYPL) is currently trading at $44.10, with a market cap of $41.258 billion. The company operates a technology platform that enables digital payments for merchants and consumers worldwide. PayPal is facing challenges with a securities lawsuit and CEO transition, which have raised concerns about its leadership and financial performance. Despite this, the company's shares remain significantly undervalued, trading well below analyst targets. PayPal's YTD return is 24.24%, outperforming the S&P 500. The company has a strong track record of profitability, with a profit margin of 15.77% and a return on equity of 25.73%. Analyst price targets for PayPal range from $32.00 to $100.00, with the average target at $50.23.

> Finished chain.
 

PayPal Holdings, Inc. (PYPL) is currently trading at $44.10, with a market cap of $41.258 billion. The company operates a technology platform that enables digital payments for merchants and consumers worldwide. PayPal is f

I need to ensure that the team develops trading strategies for PYPL that align with the user's risk tolerance and trading preferences. I must make the most of the insights provided by the Data Analyst and Trading Strategy Developer.

Action: Delegate work to co-worker
Action Input: {"coworker": "Trading Strategy Developer", "task": "Develop trading strategies for PYPL based on insights from Data Analyst and user-defined risk tolerance (Medium-Low), considering trading preferences (Day Trading)", "context": "The Data Analyst has analyzed market data for PYPL. Utilize statistical modeling and machine learning to identify trends and predict market movements."}

> Entering new CrewAgentExecutor chain...
I should start by reading the website content for PYPL to gather information on the current market trends and any recent news that may impact the stock.

Action: Read website content
Action Input: {"website_url": "https://www.paypal.com/"} 

Pay, Send and Save Money with PayPal | PayPal US


Final Answer: 
After reviewing the website content for PayPal (PYPL), it is clear that PayPal is not only a payment platform but also offers various financial services and products such as debit cards, credit cards, savings accounts, cryptocurrency trading, and installment payment options. This diversified portfolio of services indicates that PayPal is expanding its offerings beyond traditional payment processing, which could positively impact its revenue and growth potential.

Based on the insights from the Data Analyst Agent and considering the user-defined risk tolerance (Medium-Low) for novice investors interested in Day Trading, it is recommended to closely monitor market trends related to PayPal's financial services and products. Additionally, incorporating statistical modeling and machine learning techniques to identify patterns and predict market movements can help in developing effective trading strategies.

Overall, with the increasing adoption of digital payment solutions an

I need to ensure that I gather all the necessary information from my team members in order to provide a comprehensive risk analysis report for PYPL.

Action: Delegate work to co-worker
Action Input: {"coworker": "Risk Advisor", "task": "Evaluate the risks associated with the proposed trading strategies and execution plans for PYPL. Provide a detailed analysis of potential risks and suggest mitigation strategies.", "context": "Detailed execution plans have been suggested for executing trades for PYPL based on the insights provided by the Data Analyst."}

> Entering new CrewAgentExecutor chain...
I need to thoroughly evaluate the risks associated with the proposed trading strategies for PYPL and suggest appropriate mitigation strategies based on the insights provided by the Data Analyst.

Action: Read website content
Action Input: {"website_url": "https://www.investopedia.com/markets/stocks/pypl/"} 

If you are a reader experiencing an access issue, please contact
 support@people.inc. To

- Display the final result as Markdown.

In [17]:
from IPython.display import Markdown
Markdown(result)

Risk Analysis Report for PYPL:

Introduction:
PayPal Holdings Inc. (PYPL) is a leading company in the financial markets, offering innovative payment solutions and services. In this report, we will evaluate the risks associated with trading PYPL stocks and provide recommendations for mitigating these risks.

Market Volatility:
One of the key risks associated with trading PYPL stocks is market volatility. The stock price of PYPL can be influenced by various market factors, including economic conditions, geopolitical events, and investor sentiment. To mitigate this risk, it is important to diversify the trading portfolio and use risk management strategies such as stop-loss orders.

Regulatory Changes:
The fintech industry, in which PYPL operates, is subject to regulatory changes that can impact the company's operations and stock price. It is crucial to stay informed about regulatory developments and adjust trading strategies accordingly to minimize the impact of regulatory changes on PYPL stocks.

Competition:
PYPL faces competition from other payment processors and financial technology companies, which can affect its market share and profitability. To mitigate this risk, it is important to monitor the competitive landscape and adapt trading strategies based on the competitive environment.

Cybersecurity Threats:
As a leading payment solutions provider, PYPL is a target for cyber attacks that can compromise customer data and disrupt its operations. Implementing robust cybersecurity measures, such as encryption protocols and regular security audits, can help mitigate the risk of cybersecurity threats impacting PYPL stocks.

Macroeconomic Factors:
PYPL's stock price can be influenced by macroeconomic factors such as consumer spending trends, interest rates, and inflation. It is important to closely monitor economic indicators and adjust trading strategies based on macroeconomic conditions to minimize the impact of these factors on PYPL stocks.

Conclusion:
In conclusion, while PYPL presents a promising investment opportunity, it is essential to be aware of the risks associated with trading PYPL stocks and have a comprehensive risk management strategy in place. By diversifying the trading portfolio, staying informed about regulatory changes, monitoring the competitive landscape, implementing robust cybersecurity measures, and adjusting trading strategies based on macroeconomic factors, investors can navigate the potential challenges and maximize returns when trading PYPL stocks.

In [18]:
# Print the result
with open("collaboration_financial_analysis_mid-low_risk.md", "w") as f:
    f.write(result)